<a href="https://colab.research.google.com/github/alfarisauliarahman/BigData26_A_2411533006_AlfarisAuliaRahman/blob/main/Praktikum02/BD_A_P02_2411533006_AlfarisAuliaRahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 2 - Pengumpulan dan Pra-pemrosesan Data

**Nama:** Alfaris Aulia Rahman  
**NIM:** 2411533006  
**Kelas:** A

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.3 MB/s eta 0:00:00


# K. Langkah Kerja

## K-1. Import Library dan Inisialisasi

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

## K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

In [3]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga",
"Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max",
"Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000,
1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"),
tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name":
produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method",
0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


## K-3. Deteksi dan Penanganan Missing Value

In [4]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [5]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

## K-4. Deteksi dan Penanganan Duplicate

In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df["transaction_id"].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


## K-5. Koreksi Tipe Data dan Standardisasi Format

### a. Standardisasi teks kategorikal (`category`, `payment_method`, `shipping_city`)

In [7]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

### b. Koreksi tipe data pada kolom `price` (dari teks bercampur simbol menjadi numerik)

In [8]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

### c. Standardisasi format tanggal ke YYYY-MM-DD

In [9]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

### d. Finalisasi tipe data

In [10]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

## K-6. Ekspor Dataset Bersih

In [11]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


# P. Studi Kasus

### 1. Mengapa jumlah transaksi tim IT (515) berbeda dengan jumlah yang dipakai tim Finance (490)?

Tim IT menghitung seluruh baris data mentah, termasuk baris duplicate dan transaksi yang kehilangan data wajib. Setelah preprocessing, baris duplicate dihapus dan baris yang tidak memiliki `customer_name` atau `payment_method` dibuang. Akibatnya, 25 baris tidak dipakai sehingga jumlahnya berkurang dari 515 menjadi 490 baris.

### 2. Apakah 490 baris lebih benar dibandingkan 515 baris?

Untuk kebutuhan analisis Finance, 490 baris lebih dapat dipercaya karena tidak lagi memuat pencatatan ganda dan transaksi yang kehilangan informasi wajib. Hal ini meningkatkan **Veracity**, yaitu tingkat keakuratan dan keterpercayaan data. Namun, 515 tetap benar sebagai jumlah baris data mentah yang diterima sebelum proses pembersihan.

### 3. Mengapa missing value pada `rating` dibiarkan kosong?

Rating bersifat opsional karena tidak semua pembeli memberikan penilaian. Mengisi nilai kosong dengan angka tebakan dapat mengubah distribusi dan menghasilkan rata-rata yang menyesatkan. Karena itu, rata-rata rating sebaiknya dihitung hanya dari transaksi yang benar-benar memiliki rating, sambil melaporkan jumlah rating yang tersedia dan jumlah yang kosong.

# Q. Latihan

## Latihan 1

Ubah `SEED` menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris `transaksi_mentah.csv` dan `transaksi_bersih.csv` dengan hasil `SEED = 42`. Apakah jumlahnya sama? Jelaskan mengapa.

In [12]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga",
"Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max",
"Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000,
1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"),
tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name":
produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method",
0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah_7.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [13]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


In [14]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

In [15]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df["transaction_id"].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [16]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [17]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [18]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [19]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [20]:
df.to_csv("transaksi_bersih_7.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Jumlah baris data mentah tetap sama, yaitu 515 baris, dan data bersih tetap 490 baris. Perubahan `SEED` mengubah data serta baris yang terpilih secara acak, tetapi tidak mengubah jumlah data awal (500), jumlah baris yang diduplikasi (15), maupun proporsi missing value yang disuntikkan.

## Latihan 2

Tambahkan kolom `is_valid_price` bernilai `True` jika `price > 0`. Gunakan untuk memeriksa apakah ada harga tidak valid.

In [21]:
df["is_valid_price"] = df["price"] > 0

print(df["is_valid_price"].value_counts())
print("Jumlah harga tidak valid:", (~df["is_valid_price"]).sum())

is_valid_price
True    490
Name: count, dtype: int64
Jumlah harga tidak valid: 0


## Latihan 3

Hitung jumlah transaksi per `category` menggunakan `value_counts()` pada dataset yang sudah bersih.

In [22]:
print(df["category"].value_counts())

category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64
